# 1. Working with Targets

In this tutorial, you will learn how to initialize the `Astrometrics` high-level interface, fetch targets from the catalog, and inspect the `Target` domain object.


## 1.1 Initializing Astrometrics

The `Astrometrics` object is your primary entry point. It provides access to the `targets`, `processing`, `visualization`, `stars`, and `moving_objects` registries.


In [1]:
from astrometricslib import Astrometrics

astrometrics = Astrometrics()
print("Astrometrics initialized successfully.")

Astrometrics initialized successfully.


## 1.2 Loading the Bundled Sample Data

The repository includes real sample images of the Hercules Globular Cluster (**M 13**) under `documentation/notebooks/astrometrics/sample_data/` so you can follow along without your own telescope data.

These sample files contain two types of pictures of M 13:

- **Luminance frames**: Standard pictures taken through a clear filter passing visible light. These feed the imaging, plate-solving, and photometry pipelines.
- **Spectroscopy frames**: Pictures taken through a diffraction grating (filter name `SPEC`), which spreads each star's light into a spectrum to measure temperature and composition.

### One Target, Multiple Frame Types

In `astrometricslib`, a **Target** represents the physical celestial object in the sky (**M 13**). All photographs of that object belong to the same target, regardless of which filter was used.

You do not need to create separate targets for different filters. Each frame automatically records its own filter type from its FITS header. When stacking or analyzing frames in later tutorials, you simply specify which filter to process (for example, stacking only the Luminance frames or only the Spectroscopy frames).

Here, we register a single target (**M 13**) and attach all sample light frames to it:


In [2]:
# 1. Copy sample images to local working storage
from documentation.notebooks.astrometrics.scripts.sample_data_staging import (
    drop_stale_sample_data_frames,
    stage_m13_sample_data,
)

sample_data_dir = stage_m13_sample_data()

# 2. Get or create the single M 13 target
target = astrometrics.targets.get("M 13") or astrometrics.targets.create("M 13")
drop_stale_sample_data_frames(target)

# 3. Register all sample light frames (Luminance and Spectroscopy)
for path in sorted(sample_data_dir.glob("M_13_Light_*.fits")):
    if not any(f.path == str(path) for f in target.frames):
        astrometrics.targets.add_frame(target, path=str(path), role="LIGHT")

astrometrics.targets.save()
print(f"Target '{target.id}' ready with {len(target.frames)} registered frames.")

Target 'M 13' ready with 157 registered frames.


## 1.3 Fetching Targets

You can list all targets or get a specific target using the `astrometrics.targets` registry.


In [3]:
targets = astrometrics.targets.list()
print(f"Found {len(targets)} targets in the catalog.")

# Let's get a specific target, for example M 13
target = astrometrics.targets.get("M 13")
print(f"Fetched Target: {target.id}")

Found 45 targets in the catalog.
Fetched Target: M 13


## 1.4 Inspecting the Target Object

The `Target` object holds metadata and a list of `FrameRecord` objects representing the FITS files associated with this target.


In [4]:
print(f"Target ID: {target.id}")
print(f"Target Type: {target.image_type}")
print(f"Total Exposure Time: {target.exposure_sec} seconds")
print(f"Number of registered frames: {len(target.frames)}")

Target ID: M 13
Target Type: target_image
Total Exposure Time: 5940.000128 seconds
Number of registered frames: 157


## 1.5 Inspecting Frame Records

Let's look at the first few frame records attached to this target.


In [5]:
for i, frame in enumerate(target.frames[:3]):
    print(f"Frame {i + 1}:")
    print(f"  Role: {frame.role}")
    print(f"  Filter: {frame.filter}")
    print(f"  Exposure: {frame.exposure}s")
    print(f"  Path: {frame.path}")
    print("-" * 20)

Frame 1:
  Role: LIGHT
  Filter: None
  Exposure: 30.0s
  Path: /media/michael/OS/Users/macoe/Library/lights/M 13/Apertura 75Q/Nikon DSLR DSC D5300/M13_Light_017.fits
--------------------
Frame 2:
  Role: LIGHT
  Filter: None
  Exposure: 30.0s
  Path: /media/michael/OS/Users/macoe/Library/lights/M 13/Apertura 75Q/Nikon DSLR DSC D5300/M_13_Light_014.fits
--------------------
Frame 3:
  Role: LIGHT
  Filter: None
  Exposure: 30.0s
  Path: /media/michael/OS/Users/macoe/Library/lights/M 13/Apertura 75Q/Nikon DSLR DSC D5300/M13_Light_001.fits
--------------------
